# 00 — Point-in-time extract and monthly SCR vectors

**Purpose.** Freeze the application audience, render/run Athena extraction, fit a record encoder on pre-cutoff unlabeled SCR, and materialize one learned vector per application × SCR reference month. Run this notebook before 01 and 02.

The default `DEMO=True` makes synthetic data and is only an execution check. For bank data, set `DEMO=False`, fill `REAL`, install `python -m pip install -e '.[experiments,aws]'`, and use fresh empty S3 UNLOAD prefixes. Physical spine columns and eight months remain explicit in the private JSON config. Stage 0.2's availability and authorization investigation continues; the source basis is recorded in every output.

**Temporal contract.** The fit anchor and every record, history model, centroid and reference scale must predate the earliest application being scored. Each monthly vector at application `T` uses only records selected as available before `T`. No credit label trains this encoder.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd

REPO = Path.cwd()
if not (REPO / 'scr_features').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

DEMO = True
WORK = REPO / 'notebook_runs' / ('demo' if DEMO else 'real')
WORK.mkdir(parents=True, exist_ok=True)
EXECUTE_ATHENA = False
BUCKET_COUNT = 16
ANCHOR_OFFSETS = (1, 2, 3)
RUN_TAG = 'replace_with_unique_run_id'
FIT_SAMPLE_MOD = 1000  # Increase this divisor if the historical fit extract is too large.

# Fill these for the real run. Each application group is one month × hash bucket.
# Anchor groups use the same keys and can glob several as-of offsets.
REAL = {
    'config': '', 'fit_anchor': '', 'earliest_application': '',
    'fit_raw': '', 'application_groups': {}, 'anchor_groups': {},
    'incumbent': '', 'validation_month': '',
    'baseline_columns': [], 'train_provenances': ['OBSERVED'],
    's3_root': '', 'athena_workgroup': '', 'query_result_s3': '', 'region': None,
}
print('Workspace:', WORK, '| synthetic demo:', DEMO)

## A. Render Athena SQL (real mode)

The fit query samples people from consultations before `fit_anchor` without using future applicant membership. Application queries retain the full audience. Anchor queries re-extract earlier as-of states; they omit anchors at or before the representation fit cutoff. `EXECUTE_ATHENA=True` submits the generated UNLOAD statements and downloads Parquet with the AWS default credential chain. Review generated SQL and expected scan cost before enabling it. Leave it false if your team runs SQL separately; then populate local paths in `REAL`.

In [ ]:
from scr_features.sql import load_config, render_fit_sql, render_sql, render_anchor_sql
from scr_features.athena import execute_unload, download_parquet_prefix
from scr_features.demo import make_demo_data

if DEMO:
    demo = make_demo_data(str(WORK / 'raw'), applicants_per_product_month=16)
    inputs = {**demo,
              'application_groups': {'demo': demo['application_raw']},
              'anchor_groups': {'demo': demo['anchor_raw']},
              'baseline_columns': ['incumbent_score_a', 'incumbent_score_b'],
              'train_provenances': ['OBSERVED']}
else:
    cfg = load_config(REAL['config'])
    sql_dir = WORK / 'sql'
    sql_dir.mkdir(exist_ok=True)
    root = REAL['s3_root'].rstrip('/')
    if not root.startswith('s3://') or not REAL['fit_anchor']:
        raise ValueError('Set s3_root and fit_anchor in REAL')
    jobs = []
    fit_s3 = f'{root}/{RUN_TAG}/fit/'
    jobs.append(('fit', render_fit_sql(cfg, REAL['fit_anchor'], REAL['earliest_application'],
                                       fit_s3, sample_mod=FIT_SAMPLE_MOD), fit_s3))
    for month in cfg['application_months']:
        prefix = f'{root}/{RUN_TAG}/applications/month={month}/'
        jobs.append((f'application_{month}', render_sql(
            cfg, month, prefix, partitioned_buckets=BUCKET_COUNT), prefix))
        for offset in ANCHOR_OFFSETS:
            anchor_prefix = f'{root}/{RUN_TAG}/anchors/month={month}/offset={offset}/'
            jobs.append((f'anchor_{month}_o{offset}', render_anchor_sql(
                cfg, month, offset, REAL['fit_anchor'], anchor_prefix,
                partitioned_buckets=BUCKET_COUNT), anchor_prefix))
    for name, sql, _ in jobs:
        (sql_dir / f'{name}.sql').write_text(sql, encoding='utf-8')
    print(len(jobs), 'SQL files written to', sql_dir)
    if EXECUTE_ATHENA:
        if RUN_TAG == 'replace_with_unique_run_id':
            raise ValueError('Choose a fresh RUN_TAG before submitting Athena queries')
        for name, sql, prefix in jobs:
            execute_unload(sql, workgroup=REAL['athena_workgroup'],
                           query_result_s3=REAL['query_result_s3'], region=REAL['region'])
            try:
                download_parquet_prefix(prefix, str(WORK / 'downloaded' / name), region=REAL['region'])
            except ValueError as exc:
                if not name.startswith('anchor_') or 'No Parquet files' not in str(exc):
                    raise  # An early anchor may validly have no post-fit-cutoff rows.
    inputs = {**REAL}
    if EXECUTE_ATHENA:
        inputs['fit_raw'] = str(WORK / 'downloaded' / 'fit' / '*.parquet')
        inputs['application_groups'] = {
            f'{month}_b{bucket:02d}': str(WORK / 'downloaded' / f'application_{month}' / f'bucket={bucket}' / '*.parquet')
            for month in cfg['application_months'] for bucket in range(BUCKET_COUNT)
            if list((WORK / 'downloaded' / f'application_{month}' / f'bucket={bucket}').glob('*.parquet'))}
        inputs['anchor_groups'] = {
            f'{month}_b{bucket:02d}': str(WORK / 'downloaded' / f'anchor_{month}_o*' / f'bucket={bucket}' / '*.parquet')
            for month in cfg['application_months'] for bucket in range(BUCKET_COUNT)
            if list((WORK / 'downloaded').glob(f'anchor_{month}_o*/bucket={bucket}/*.parquet'))}
    if not inputs['fit_raw'] or not inputs['application_groups'] or not inputs['incumbent']:
        raise ValueError('Set the local fit, application and incumbent paths in REAL after Athena extraction')
print('Application extract groups:', len(inputs['application_groups']))

## B. Fit the shared record-to-month encoder

The small record MLP consumes current and overdue balances plus verified modality/institution codes. Within-month pooling yields `z` and a masked-record reconstruction task trains it without credit labels. This is a separate model from the later history aggregators. In production, increase epochs only through the pre-registered training budget and inspect reconstruction loss.

In [ ]:
from scr_features.hierarchy import fit_record_encoder, encode_months

record_dir = WORK / 'models' / 'record_month'
record_manifest = fit_record_encoder(
    inputs['fit_raw'], inputs['earliest_application'], str(record_dir),
    month_dim=16, epochs=2 if DEMO else 25, batch_months=128 if DEMO else 256)
print({k: record_manifest[k] for k in ('version', 'fit_max_anchor', 'monthly_fit_rows', 'final_reconstruction_loss')})

## C. Materialize monthly vectors and preserve all applications

Each extract group is scored separately to bound memory. `apps.parquet` retains covered and uncovered applications; `months.parquet` contains only observed reference months. The same frozen record encoder scores the pre-cutoff fit corpus, the eight application months, and earlier anchor extracts. Anchor extracts are needed only for the separate B5 as-of trajectory.

In [ ]:
fit_months = WORK / 'vectors' / 'fit_months.parquet'
fit_apps = WORK / 'vectors' / 'fit_apps.parquet'
encode_months(inputs['fit_raw'], str(record_dir), str(fit_months), str(fit_apps), for_fit=True)

groups = {}
for key, raw in inputs['application_groups'].items():
    month_path = WORK / 'vectors' / key / 'months.parquet'
    app_path = WORK / 'vectors' / key / 'apps.parquet'
    encode_months(raw, str(record_dir), str(month_path), str(app_path))
    groups[key] = {'raw': raw, 'months': str(month_path), 'apps': str(app_path)}

anchors = {}
for key, raw in inputs['anchor_groups'].items():
    month_path = WORK / 'vectors' / f'anchor_{key}' / 'months.parquet'
    app_path = WORK / 'vectors' / f'anchor_{key}' / 'apps.parquet'
    encode_months(raw, str(record_dir), str(month_path), str(app_path))
    anchors[key] = {'raw': raw, 'months': str(month_path), 'apps': str(app_path)}

stage00 = {'demo': DEMO, 'fit_months': str(fit_months), 'fit_apps': str(fit_apps),
           'record_model': str(record_dir), 'groups': groups, 'anchors': anchors,
           'incumbent': inputs['incumbent'], 'earliest_application': inputs['earliest_application'],
           'validation_month': inputs['validation_month'],
           'baseline_columns': inputs['baseline_columns'],
           'train_provenances': inputs['train_provenances']}
(WORK / 'stage00.json').write_text(json.dumps(stage00, indent=2), encoding='utf-8')
print('Applications:', sum(len(pd.read_parquet(v['apps'])) for v in groups.values()),
      '| monthly vectors:', sum(len(pd.read_parquet(v['months'])) for v in groups.values()))

**Check before proceeding:** compare the summed application count with the frozen spine, inspect coverage by product/month/customer status, and investigate any missing bucket or duplicate key. Do not interpret synthetic demo numbers as evidence.